# Browsing a sweep

`box.ipynb` and `column.ipynb` browse **one** CrunchTope or MIN3P run. This browses a **sweep** -- many runs
that differ in whatever was varied -- and the difference matters for how it is laid out.

A sweep is run to compare across runs, so the run axis is drawn **whole**: every plot here shows all
the runs at once and the widgets choose *which variable* to look at, not *which run*. Stepping
through runs one at a time would hide the very thing the sweep was run to show. The exception is the
2-D map at the end, which can only show one run.

**The widgets are a convenience, not a dependency.** Every control below does nothing but call a
function from `coeus.sweep_plots`. If the widget stack misbehaves, call those functions directly --
the last cell shows how -- and you lose the sliders, not the analysis.

## Canary

Run this first. If you do not see a slider, the widget stack is not working in this kernel and
nothing below will draw. Skip to the last cell and call the plotting functions directly.

Widgets need `ipywidgets` **and** a kernel that can render it. Use the `JupyterEnv` kernel: the
`topepan` environment has ipywidgets but no `ipykernel`, so it cannot act as one.

In [ ]:
import ipywidgets as widgets

print(f'ipywidgets {widgets.__version__}')
widgets.IntSlider(description='canary')

## Load a sweep

`describe` is worth reading before any plotting. It says how many runs there are, what was varied,
and -- the part that is easy to miss -- whether the runs actually finished. A sweep whose runs time
out still writes a `results.nc` full of plausible numbers.

In [ ]:
%matplotlib ipympl
import os
import sys
from pathlib import Path


def find_omphalos():
    """Locate the Omphalos checkout so `coeus` can be imported.

    Not a fixed path: topepan is used on more than one machine and Omphalos does not always sit in
    the same place. Preference order is an explicit OMPHALOS_DIR, then a sibling directory of
    wherever this notebook is running, which is the usual layout -- topepan/ and Omphalos/ next to
    each other in a CrunchTope working directory.

    Returns None if Omphalos is already importable, e.g. `pip install -e` into this environment.
    """
    try:
        import coeus.sweep  # noqa: F401
        return None
    except ImportError:
        pass

    if os.environ.get('OMPHALOS_DIR'):
        return Path(os.environ['OMPHALOS_DIR'])

    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'Omphalos'
        if (candidate / 'coeus' / 'sweep.py').is_file():
            return candidate

    raise ImportError(
        'Could not find Omphalos, which sweep.ipynb needs for coeus. Either set OMPHALOS_DIR to '
        'the checkout, or install it into this environment with `pip install -e /path/to/Omphalos`.')


found = find_omphalos()

if found is not None:
    sys.path.insert(0, str(found))
    print(f'using Omphalos at {found}')

import matplotlib.pyplot as plt
from coeus.sweep import Sweep, describe
from coeus import sweep_plots as sp


def new_panel():
    """A figure of its own, plus a redraw that clears and returns a fresh axes.

    Each browser below gets its own, because `fig` and `redraw` would otherwise be globals that
    every cell rebinds -- the last cell run would own them, and the profiles widget would redraw
    into the field cell's figure.

    The whole figure is cleared rather than just the axes: `field` attaches a colorbar, and
    clearing only the axes leaves the old one behind to stack another beside it on every redraw.
    constrained_layout leaves room for the legend, which sits outside the axes.
    """
    figure = plt.figure(constrained_layout=True)

    def redraw():
        figure.clear()
        return figure.add_subplot()

    return figure, redraw

sp.use_style()

# The sweep to read: either the results.nc, or the directory holding it. Whatever records
# what varied -- conditions.nc from CrunchTope, records.pkl from MIN3P -- is found alongside.
RESULTS = Path('/path/to/your/sweep/results.nc')

sweep = Sweep(RESULTS)
describe(sweep)

## Profiles along the column

One line per run, at the chosen output time. The legend names whatever was swept, so the lines say
what distinguishes them rather than merely being numbered.

Where a sweep crosses two parameters, the labels default to whichever separates the most runs. If
that is not the one you care about, pass `parameter=` to `sp.profiles` directly.

In [ ]:
# Asked of the sweep rather than matched by name, so this works whichever code wrote it:
# CrunchTope names its axes X/Y/Z and MIN3P names them x/y/z, and neither records which it is.
spatial = sweep.spatial_groups()


profile_fig, profile_redraw = new_panel()


def show_profiles(group, variable, time, orientation):
    sp.profiles(sweep, group, variable, time=time, axis=profile_redraw(),
                vertical=orientation)
    profile_fig.canvas.draw_idle()


group_pick = widgets.Dropdown(options=sorted(spatial), description='group')
variable_pick = widgets.Dropdown(options=sorted(sweep.data(group_pick.value).data_vars),
                                 description='variable')
# X down the y axis is the depth convention; X across the x axis reads better for a flow
# path. Both are useful for a 1-D column.
orientation_pick = widgets.ToggleButtons(
    options=[('distance on x axis', False), ('distance on y axis (depth)', True)],
    value=False, description='Orientation')
# Snapshots, not times: MIN3P indexes its spatial output by a counter with no simulated time
# attached to it, so the slider counts what was written rather than claiming a time.
snapshots = sweep.snapshot_count(group_pick.value)
time_pick = widgets.IntSlider(min=0, max=snapshots - 1, value=snapshots - 1,
                              description='snapshot')


def on_group(change):
    # The slider is re-ranged first. Setting the variable list redraws straight away, so doing that
    # while the slider still holds the old group's snapshot index asks for a snapshot the new group
    # may not have -- MIN3P writes its velocity field once and its chemistry nine times.
    snapshots = sweep.snapshot_count(change['new'])
    time_pick.max = max(time_pick.max, snapshots - 1)
    time_pick.value = min(time_pick.value, snapshots - 1)
    time_pick.max = snapshots - 1
    variable_pick.options = sorted(sweep.data(change['new']).data_vars)


group_pick.observe(on_group, names='value')
widgets.interact(show_profiles, group=group_pick, variable=variable_pick, time=time_pick,
                 orientation=orientation_pick);

## Time series at an observation point

The `timeseries_*` groups are written every timestep rather than at the snapshot times, so they show
arrival and breakthrough that the profiles above cannot resolve. Runs that stopped early simply end
early rather than running a flat line to the edge of the axis.

In [ ]:
# CrunchTope calls these timeseries_*, MIN3P calls them gbc/gbm/gbt. Both give them a `time`
# coordinate over `step`, which is what series_groups() looks for.
series_groups = sweep.series_groups()

if series_groups:
    series_fig, series_redraw = new_panel()

    def show_series(group, variable):
        sp.time_series(sweep, group, variable, axis=series_redraw())
        series_fig.canvas.draw_idle()

    series_pick = widgets.Dropdown(options=sorted(series_groups), description='group')
    series_var = widgets.Dropdown(options=sorted(sweep.data(series_pick.value).data_vars),
                                  description='variable')

    series_pick.observe(
        lambda change: setattr(series_var, 'options',
                               sorted(sweep.data(change['new']).data_vars)),
        names='value')

    widgets.interact(show_series, group=series_pick, variable=series_var);
else:
    print('this sweep records nothing through time at a fixed point')

## One run at a time: the 2-D map

The only view that cannot show the sweep axis whole, so here the run *is* the control. On a 1-D
column this is not worth looking at, and the cell says so rather than drawing a stripe.

In [ ]:
two_d = sweep.map_groups()

if two_d:
    field_fig, field_redraw = new_panel()

    def show_field(group, variable, run, time):
        sp.field(sweep, group, variable, run=run, time=time, axis=field_redraw())
        field_fig.canvas.draw_idle()

    field_group = widgets.Dropdown(options=sorted(two_d), description='group')
    field_var = widgets.Dropdown(options=sorted(sweep.data(field_group.value).data_vars),
                                 description='variable')
    run_pick = widgets.IntSlider(min=min(sweep.runs), max=max(sweep.runs), description='run')

    field_time = widgets.IntSlider(min=0, max=sweep.snapshot_count(field_group.value) - 1,
                                   value=sweep.snapshot_count(field_group.value) - 1,
                                   description='snapshot')


    def on_field_group(change):
        # Same ordering as the profiles browser above, and for the same reason.
        snapshots = sweep.snapshot_count(change['new'])
        field_time.max = max(field_time.max, snapshots - 1)
        field_time.value = min(field_time.value, snapshots - 1)
        field_time.max = snapshots - 1
        field_var.options = sorted(sweep.data(change['new']).data_vars)


    field_group.observe(on_field_group, names='value')

    widgets.interact(show_field, group=field_group, variable=field_var, run=run_pick,
                     time=field_time);
else:
    print(f'{RESULTS.name} is 1-D, so there is no 2-D field to map')

## Without the widgets

Everything above is one call into `coeus.sweep_plots`. If the widget stack is broken, or you are
working from a script rather than a notebook, call them directly -- and this is also the starting
point for a figure that is going into something.

In [ ]:
# Names taken from the sweep rather than written in, so this runs against whichever one is loaded:
# 'totcon' and 'SO4--' exist in ex9 and not in ex8, whose totals are Ca++ and Ca44++.
group = sorted(g for g in spatial)[0]
variable = sorted(sweep.data(group).data_vars)[0]
series = next(iter(sweep.series_groups()), None)

panels = 2 if series else 1
fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 3.4), constrained_layout=True,
                         squeeze=False)

sp.profiles(sweep, group, variable, axis=axes[0][0])

if series:
    sp.time_series(sweep, series, sorted(sweep.data(series).data_vars)[0],
                   axis=axes[0][1], legend=False)

sp.label_panels(axes)
fig.savefig('sweep_overview.png', dpi=200)
print(f'plotted {variable} from {group}' + (f', and {series}' if series else ''))